# 01 — Exploratory Data Analysis
**AI Cost Optimization Portfolio**

Explores the synthetic cloud billing dataset.
---

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
plt.rcParams.update({'figure.dpi':120,'figure.figsize':(12,5),'axes.titlesize':14,'axes.titleweight':'bold'})
USD = mticker.FuncFormatter(lambda x,_: f'${x:,.0f}')
print('Libraries loaded OK')

## 1 - Load Data

In [ ]:
DATA_PATH = Path('../data/raw/billing_data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Not found: {DATA_PATH}. Run synth_generator.py first.')
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head()

## 2 - Dataset Overview

In [ ]:
print('=== dtypes ==='); print(df.dtypes,'\n')
print('=== missing values ==='); print(df.isnull().sum(),'\n')
print('=== numeric summary ==='); df.describe().round(2)

In [ ]:
print(f"Date range : {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"Services   : {sorted(df['service'].unique())}")
print(f"Teams      : {sorted(df['team'].unique())}")
n_anom = df['anomaly_flag'].sum()
anom_spend = df.loc[df['anomaly_flag'],'total_cost'].sum()
print(f'Total rows : {len(df):,}')
print(f'Anomalies  : {n_anom:,}  ({100*n_anom/len(df):.2f}%)')
print(f'Total spend: ${df["total_cost"].sum():,.2f}')
print(f'Anom spend : ${anom_spend:,.2f}  ({100*anom_spend/df["total_cost"].sum():.1f}% of total)')

## 3 - Cost Distribution

In [ ]:
Path('../data/processed').mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(14,5))
ax = axes[0]
ax.hist(df.loc[~df['anomaly_flag'],'total_cost'], bins=80, color='steelblue', alpha=0.7, label='Normal', log=True)
ax.hist(df.loc[df['anomaly_flag'],'total_cost'],  bins=80, color='tomato',    alpha=0.8, label='Anomaly', log=True)
ax.set_xlabel('Total Cost (USD)'); ax.set_ylabel('Count (log)'); ax.set_title('Cost Distribution')
ax.legend(); ax.xaxis.set_major_formatter(USD)
ax = axes[1]
order = df.groupby('service')['total_cost'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='service', y='total_cost', order=order, palette='tab10', showfliers=False, ax=ax)
ax.set_title('Cost by Service (IQR)'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30)
ax.yaxis.set_major_formatter(USD)
plt.tight_layout()
plt.savefig('../data/processed/cost_distribution.png', bbox_inches='tight'); plt.show()

## 4 - Spend by Service, Region and Team

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
for col, title, ax in [('service','By Service',axes[0]),('region','By Region',axes[1]),('team','By Team',axes[2])]:
    data = df.groupby(col)['total_cost'].sum().sort_values()
    data.plot(kind='barh', ax=ax, color=sns.color_palette('tab10', len(data)), edgecolor='white')
    ax.set_title(f'Total Spend {title}'); ax.xaxis.set_major_formatter(USD)
plt.tight_layout()
plt.savefig('../data/processed/spend_breakdown.png', bbox_inches='tight'); plt.show()

## 5 - Daily Spend Trends

In [ ]:
daily = df.set_index('timestamp').resample('D')['total_cost'].sum().to_frame('daily_cost')
daily['r7']  = daily['daily_cost'].rolling(7,  min_periods=1).mean()
daily['r14'] = daily['daily_cost'].rolling(14, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(14,5))
ax.fill_between(daily.index, daily['daily_cost'], alpha=0.25, color='steelblue', label='Daily')
ax.plot(daily.index, daily['r7'],  color='darkorange', lw=2, label='7-day avg')
ax.plot(daily.index, daily['r14'], color='firebrick',  lw=2, label='14-day avg', ls='--')
ax.set_title('Daily Cloud Spend - Last 90 Days'); ax.set_xlabel('Date')
ax.yaxis.set_major_formatter(USD); ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/daily_trend.png', bbox_inches='tight'); plt.show()

## 6 - Anomaly Profile

In [ ]:
anom = df.groupby('service').agg(
    total_records=('record_id','count'),
    anomaly_count=('anomaly_flag','sum'),
    total_cost=('total_cost','sum'),
    anomaly_cost=('total_cost', lambda x: x[df.loc[x.index,'anomaly_flag']].sum()),
).assign(
    anomaly_rate_pct=lambda d: 100*d['anomaly_count']/d['total_records'],
    anomaly_cost_pct=lambda d: 100*d['anomaly_cost']/d['total_cost'],
).sort_values('anomaly_cost_pct', ascending=False)
display(anom.round(2))
fig, axes = plt.subplots(1, 2, figsize=(14,5))
anom['anomaly_rate_pct'].sort_values().plot(kind='barh', ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('Anomaly Rate by Service (%)')
anom['anomaly_cost_pct'].sort_values().plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Anomaly Cost Share (%)')
plt.tight_layout()
plt.savefig('../data/processed/anomaly_profile.png', bbox_inches='tight'); plt.show()

## 7 - Correlation Heatmap

In [ ]:
corr = df[['usage_hours','unit_cost','total_cost','anomaly_flag']].corr()
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', bbox_inches='tight'); plt.show()

## 8 - Top-10 Cost Drivers

In [ ]:
top = df.groupby(['team','service','resource_type'])['total_cost'].sum()\
        .reset_index().sort_values('total_cost', ascending=False).head(10).reset_index(drop=True)
top['rank'] = top.index + 1
top['cost_fmt'] = top['total_cost'].apply(lambda x: f'${x:,.2f}')
display(top[['rank','team','service','resource_type','cost_fmt']])
fig, ax = plt.subplots(figsize=(12,5))
labels = top.apply(lambda r: r['team']+'\n'+r['resource_type'], axis=1)
bars = ax.bar(labels, top['total_cost'], color=sns.color_palette('tab10',10), edgecolor='white')
ax.set_title('Top 10 Cost Drivers'); ax.yaxis.set_major_formatter(USD)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../data/processed/top10_cost_drivers.png', bbox_inches='tight'); plt.show()

## 9 - Save EDA Summary

In [ ]:
summary = df.groupby(['service','team','region']).agg(
    records=('record_id','count'), total_cost=('total_cost','sum'),
    mean_cost=('total_cost','mean'), anomalies=('anomaly_flag','sum')
).reset_index()
out = Path('../data/processed/eda_summary.csv')
out.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out, index=False)
print(f'Saved EDA summary --> {out.resolve()}')
print(f'Shape: {summary.shape}')
summary.head(10)